In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:

# ================================================================
# DEMENTIA MRI CLASSIFICATION & ADVERSARIAL ROBUSTNESS
# CNN + FGSM + ART ADVERSARIAL TRAINING + t-SNE
# GOOGLE COLAB | TESLA T4 GPU | SINGLE CELL
# ================================================================

# 1. INSTALL DEPENDENCIES
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "adversarial-robustness-toolbox",
    "seaborn",
    "tqdm"
])

# 2. IMPORTS
import os
import gc
import json
import random
import warnings
import zipfile

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.manifold import TSNE

from google.colab import files, drive

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod
from art.defences.trainer import AdversarialTrainer

warnings.filterwarnings("ignore")

# 3. CONFIGURATION
SEED = 42

BATCH_SIZE = 32
EPOCHS = 10
ADV_EPOCHS = 10

LEARNING_RATE = 0.001

IMAGE_HEIGHT = 64
IMAGE_WIDTH = 128

EPSILON = 0.1
ADV_RATIO = 0.5

MAX_TSNE_SAMPLES = 3000

ZIP_PATH = "/content/dataset.zip"
EXTRACT_PATH = "/content/mri_dataset"

OUTPUT_DIR = (
    "/content/drive/MyDrive/Dementia_MRI_Results"
)

# 4. REPRODUCIBILITY
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 5. GPU DETECTION
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 65)
print("DEMENTIA MRI CLASSIFICATION")
print("=" * 65)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1e9,
            2
        ),
        "GB"
    )
else:
    print("WARNING: GPU not detected. Enable T4 GPU.")

# 6. CHECK UPLOADED DATASET
print("\nChecking uploaded dataset...")

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(
        "dataset.zip is not present in /content. "
        "Run the upload cell first."
    )

print("Dataset ZIP found:", ZIP_PATH)

# 7. GOOGLE DRIVE MOUNT
print("\nConnecting Google Drive...")

if os.path.isdir("/content/drive/MyDrive"):
    DRIVE_ROOT = "/content/drive"
    print("Google Drive already mounted.")
else:
    mount_path = "/content/drive"

    if os.path.exists(mount_path) and os.listdir(mount_path):
        mount_path = "/content/gdrive"

    if os.path.exists(mount_path) and os.listdir(mount_path):
        raise RuntimeError(
            "Could not find an empty Drive mount directory. "
            "Restart the Colab runtime and run again."
        )

    os.makedirs(mount_path, exist_ok=True)

    drive.mount(mount_path)
    DRIVE_ROOT = mount_path

OUTPUT_DIR = os.path.join(
    DRIVE_ROOT,
    "MyDrive",
    "Dementia_MRI_Results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Results will be saved to:", OUTPUT_DIR)

# 8. EXTRACT DATASET
print("\nExtracting dataset...")

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    archive.extractall(EXTRACT_PATH)

print("Extraction completed.")

# Locate the two class folders.
dataset_root = None

for current_root, dirs, file_names in os.walk(EXTRACT_PATH):

    folder_names = {
        folder.strip().lower()
        for folder in dirs
    }

    if (
        "dementia" in folder_names
        and "no dementia" in folder_names
    ):
        dataset_root = current_root
        break

if dataset_root is None:
    raise FileNotFoundError(
        "Could not locate Dementia and No Dementia folders. "
        "Check the ZIP folder structure."
    )

print("Dataset root:", dataset_root)

# 9. IMAGE TRANSFORMS
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )
])

dataset = datasets.ImageFolder(
    root=dataset_root,
    transform=transform
)

class_names = dataset.classes
num_classes = len(class_names)

if num_classes != 2:
    raise ValueError(
        f"Expected two classes. Found: {class_names}"
    )

print("\nClass mapping:", dataset.class_to_idx)
print("Total images:", len(dataset))

counts = Counter(dataset.targets)

for index, name in enumerate(class_names):
    print(f"{name}: {counts[index]} images")

# 10. TRAIN-TEST SPLIT
indices = np.arange(len(dataset))
labels = np.asarray(dataset.targets)

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)

train_dataset = Subset(dataset, train_indices)
test_dataset = Subset(dataset, test_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

train_eval_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("\nTraining images:", len(train_dataset))
print("Testing images:", len(test_dataset))

# Calculate class weights using training labels only.
train_labels = labels[train_indices]

class_counts = np.bincount(
    train_labels,
    minlength=num_classes
)

weights = len(train_labels) / (
    num_classes * np.maximum(class_counts, 1)
)

class_weights = torch.tensor(
    weights,
    dtype=torch.float32,
    device=DEVICE
)

# 11. CNN ARCHITECTURE
class SimpleCNN(nn.Module):

    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(8192, 128)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, num_classes)

    def extract_features(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.fc1(x)
        return torch.relu(x)

    def forward(self, x):
        x = self.extract_features(x)
        x = self.dropout(x)
        return self.fc2(x)


model = SimpleCNN(num_classes).to(DEVICE)

print("\nCNN architecture:")
print(model)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# 12. TRAINING FUNCTION
def train_model(
    model,
    loader,
    epochs,
    optimizer,
    criterion,
    title
):

    history = {
        "loss": [],
        "accuracy": []
    }

    print("\n" + "=" * 65)
    print(title)
    print("=" * 65)

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        progress = tqdm(
            loader,
            desc=f"Epoch {epoch + 1}/{epochs}"
        )

        for images, targets in progress:

            images = images.to(DEVICE)
            targets = targets.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(images)
            loss = criterion(outputs, targets)

            loss.backward()
            optimizer.step()

            running_loss += (
                loss.item() * images.size(0)
            )

            predictions = outputs.argmax(dim=1)

            correct += (
                predictions == targets
            ).sum().item()

            total += targets.size(0)

            progress.set_postfix(loss=loss.item())

        epoch_loss = running_loss / total
        epoch_accuracy = correct / total

        history["loss"].append(epoch_loss)
        history["accuracy"].append(epoch_accuracy)

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Accuracy: {epoch_accuracy:.4f}"
        )

    return history

# 13. EVALUATION FUNCTION
def evaluate_model(model, loader, title):

    model.eval()

    all_labels = []
    all_predictions = []

    total_loss = 0.0
    total = 0

    with torch.no_grad():

        for images, targets in loader:

            images = images.to(DEVICE)
            targets = targets.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, targets)

            predictions = outputs.argmax(dim=1)

            total_loss += (
                loss.item() * images.size(0)
            )

            total += targets.size(0)

            all_labels.extend(
                targets.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision = precision_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    print("\n" + title)
    print("-" * 55)

    print("Loss:", round(total_loss / total, 4))
    print("Accuracy:", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1 Score:", round(f1, 4))

    print("\nClassification Report:")

    print(classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        zero_division=0
    ))

    cm = confusion_matrix(
        all_labels,
        all_predictions,
        labels=list(range(num_classes))
    )

    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.title(title + " - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

    return {
        "loss": total_loss / total,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "labels": np.asarray(all_labels),
        "predictions": np.asarray(all_predictions)
    }

# 14. INITIAL CNN TRAINING
initial_history = train_model(
    model,
    train_loader,
    EPOCHS,
    optimizer,
    criterion,
    "INITIAL CNN TRAINING"
)

initial_train_results = evaluate_model(
    model,
    train_eval_loader,
    "Initial CNN - Training"
)

initial_test_results = evaluate_model(
    model,
    test_loader,
    "Initial CNN - Test"
)

# 15. TRAINING CURVES
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(initial_history["loss"], marker="o")
plt.title("Initial CNN - Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.subplot(1, 2, 2)
plt.plot(initial_history["accuracy"], marker="o")
plt.title("Initial CNN - Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.tight_layout()
plt.show()

# 16. LATENT FEATURE EXTRACTION
def extract_latent_features(model, loader):

    model.eval()

    features = []
    feature_labels = []

    with torch.no_grad():

        for images, targets in tqdm(
            loader,
            desc="Extracting features"
        ):

            images = images.to(DEVICE)

            latent = model.extract_features(images)

            features.append(latent.cpu().numpy())
            feature_labels.append(targets.numpy())

    return (
        np.concatenate(features, axis=0),
        np.concatenate(feature_labels, axis=0)
    )


def plot_tsne(features, labels, title):

    if len(features) > MAX_TSNE_SAMPLES:

        rng = np.random.default_rng(SEED)

        selected = rng.choice(
            len(features),
            MAX_TSNE_SAMPLES,
            replace=False
        )

        features = features[selected]
        labels = labels[selected]

    perplexity = min(
        30,
        max(2, (len(features) - 1) // 3)
    )

    print("\nRunning t-SNE:", title)

    tsne = TSNE(
        n_components=2,
        random_state=SEED,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto"
    )

    reduced = tsne.fit_transform(features)

    plt.figure(figsize=(8, 6))

    for class_index, class_name in enumerate(class_names):

        mask = labels == class_index

        plt.scatter(
            reduced[mask, 0],
            reduced[mask, 1],
            label=class_name,
            alpha=0.6,
            s=15
        )

    plt.title(title)
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.legend()
    plt.tight_layout()
    plt.show()


print("\nExtracting initial latent features...")

initial_features, initial_feature_labels = (
    extract_latent_features(model, test_loader)
)

plot_tsne(
    initial_features,
    initial_feature_labels,
    "Initial CNN - Latent Feature Visualization"
)

# 17. PREPARE ART DATA
def loader_to_numpy(loader):

    x_parts = []
    y_parts = []

    for images, targets in tqdm(
        loader,
        desc="Preparing ART data"
    ):

        x_parts.append(images.numpy())
        y_parts.append(targets.numpy())

    x = np.concatenate(
        x_parts,
        axis=0
    ).astype(np.float32)

    y = np.concatenate(
        y_parts,
        axis=0
    ).astype(np.int64)

    return x, y


print("\nPreparing Adversarial Robustness Toolbox...")

x_train_art, y_train_art = loader_to_numpy(
    train_eval_loader
)

x_test_art, y_test_art = loader_to_numpy(
    test_loader
)

y_train_onehot = np.eye(
    num_classes,
    dtype=np.float32
)[y_train_art]

# 18. ART CLASSIFIER
art_optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

art_classifier = PyTorchClassifier(
    model=model,
    loss=nn.CrossEntropyLoss(),
    optimizer=art_optimizer,
    input_shape=(
        1,
        IMAGE_HEIGHT,
        IMAGE_WIDTH
    ),
    nb_classes=num_classes,
    clip_values=(-1.0, 1.0),
    device_type=(
        "gpu" if torch.cuda.is_available() else "cpu"
    )
)

# 19. FGSM BEFORE ADVERSARIAL TRAINING
print("\nGenerating FGSM examples BEFORE adversarial training...")

fgsm_before_attack = FastGradientMethod(
    estimator=art_classifier,
    eps=EPSILON,
    batch_size=BATCH_SIZE
)

x_test_adv_before = fgsm_before_attack.generate(
    x=x_test_art
)

pred_before = np.argmax(
    art_classifier.predict(x_test_adv_before),
    axis=1
)

fgsm_before_accuracy = accuracy_score(
    y_test_art,
    pred_before
)

print(
    "FGSM accuracy before training:",
    round(fgsm_before_accuracy, 4)
)

# 20. ART ADVERSARIAL TRAINING
print("\n" + "=" * 65)
print("ART ADVERSARIAL TRAINING")
print("=" * 65)

print("FGSM epsilon:", EPSILON)
print("Adversarial ratio:", ADV_RATIO)
print("Adversarial epochs:", ADV_EPOCHS)

training_attack = FastGradientMethod(
    estimator=art_classifier,
    eps=EPSILON,
    batch_size=BATCH_SIZE
)

adversarial_trainer = AdversarialTrainer(
    classifier=art_classifier,
    attacks=training_attack,
    ratio=ADV_RATIO
)

adversarial_trainer.fit(
    x_train_art,
    y_train_onehot,
    batch_size=BATCH_SIZE,
    nb_epochs=ADV_EPOCHS
)

print("\nAdversarial training completed.")

# 21. EVALUATE ADVERSARIALLY TRAINED MODEL
after_train_results = evaluate_model(
    model,
    train_eval_loader,
    "Adversarially Trained CNN - Training"
)

after_test_results = evaluate_model(
    model,
    test_loader,
    "Adversarially Trained CNN - Test"
)

# 22. FGSM AFTER ADVERSARIAL TRAINING
print("\nGenerating FGSM examples AFTER adversarial training...")

fgsm_after_attack = FastGradientMethod(
    estimator=art_classifier,
    eps=EPSILON,
    batch_size=BATCH_SIZE
)

x_test_adv_after = fgsm_after_attack.generate(
    x=x_test_art
)

pred_after = np.argmax(
    art_classifier.predict(x_test_adv_after),
    axis=1
)

fgsm_after_accuracy = accuracy_score(
    y_test_art,
    pred_after
)

print(
    "FGSM accuracy after training:",
    round(fgsm_after_accuracy, 4)
)

# 23. ROBUSTNESS COMPARISON
comparison = {
    "Initial Clean Accuracy":
        initial_test_results["accuracy"],

    "Adversarial Clean Accuracy":
        after_test_results["accuracy"],

    "FGSM Before Training":
        fgsm_before_accuracy,

    "FGSM After Training":
        fgsm_after_accuracy
}

print("\n" + "=" * 65)
print("ROBUSTNESS COMPARISON")
print("=" * 65)

for name, value in comparison.items():
    print(f"{name}: {value:.4f}")

plt.figure(figsize=(10, 5))

plt.bar(
    list(comparison.keys()),
    list(comparison.values())
)

plt.title("CNN Performance and Adversarial Robustness")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")

plt.tight_layout()
plt.show()

# 24. FINAL LATENT FEATURES
print("\nExtracting final latent features...")

final_features, final_feature_labels = (
    extract_latent_features(model, test_loader)
)

plot_tsne(
    final_features,
    final_feature_labels,
    "Adversarially Trained CNN - Latent Feature Visualization"
)

# 25. SAVE MODEL
model_path = os.path.join(
    OUTPUT_DIR,
    "dementia_mri_adversarial_cnn.pth"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_to_idx": dataset.class_to_idx,
        "image_height": IMAGE_HEIGHT,
        "image_width": IMAGE_WIDTH,
        "epsilon": EPSILON,
        "adversarial_ratio": ADV_RATIO
    },
    model_path
)

# 26. SAVE RESULTS
results_to_save = {
    "initial_test": {
        key: float(initial_test_results[key])
        for key in [
            "loss", "accuracy", "precision", "recall", "f1"
        ]
    },

    "adversarial_test": {
        key: float(after_test_results[key])
        for key in [
            "loss", "accuracy", "precision", "recall", "f1"
        ]
    },

    "fgsm_before_accuracy": float(fgsm_before_accuracy),
    "fgsm_after_accuracy": float(fgsm_after_accuracy),

    "comparison": {
        key: float(value)
        for key, value in comparison.items()
    }
}

results_path = os.path.join(
    OUTPUT_DIR,
    "performance_results.json"
)

with open(results_path, "w") as f:
    json.dump(results_to_save, f, indent=4)

history_path = os.path.join(
    OUTPUT_DIR,
    "initial_training_history.json"
)

with open(history_path, "w") as f:
    json.dump(initial_history, f, indent=4)

# 27. FINAL SUMMARY
print("\n" + "=" * 65)
print("PROJECT EXECUTION COMPLETED")
print("=" * 65)

print("Dataset images:", len(dataset))
print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))

print(
    "Initial clean test accuracy:",
    round(initial_test_results["accuracy"], 4)
)

print(
    "Adversarial clean test accuracy:",
    round(after_test_results["accuracy"], 4)
)

print(
    "FGSM accuracy before training:",
    round(fgsm_before_accuracy, 4)
)

print(
    "FGSM accuracy after training:",
    round(fgsm_after_accuracy, 4)
)

print("\nSaved model:", model_path)
print("Saved results:", results_path)
print("Saved history:", history_path)

print("\nThis is an experimental MRI classification project.")
print("Results do not establish clinical diagnostic performance.")

# 28. CLEANUP
del x_train_art
del x_test_art
del x_test_adv_before
del x_test_adv_after

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nALL PROJECT STAGES FINISHED.")
